# DMA loopback under PYNQ

Loads `dma_loop.bit` / `dma_loop.hwh` (same design as `dma_bist`: AXI DMA MM2S -> VHDL loopback -> S2MM), then:
1. random-data BIST
2. a real webcam frame through the loopback

Transfers are split into chunks of `dma.buffer_max_size`, so this works with the 14-bit length register (16 KB) and, unchanged, with a wider one (whole frame in one transfer).

In [ ]:
import time
import numpy as np
import cv2
from pynq import Overlay, allocate

ol = Overlay('dma_loop.bit')
dma = ol.axi_dma_0
print('IP blocks :', list(ol.ip_dict.keys()))
print('max bytes per transfer :', dma.buffer_max_size)

In [ ]:
def dma_loop(tx, rx):
    """Send tx through the loopback into rx, in chunks. Returns elapsed seconds."""
    n = tx.nbytes
    chunk = (dma.buffer_max_size // 64) * 64
    tx.flush()
    rx.flush()
    t0 = time.perf_counter()
    for off in range(0, n, chunk):
        size = min(chunk, n - off)
        dma.recvchannel.transfer(rx, start=off, nbytes=size)  # arm S2MM first
        dma.sendchannel.transfer(tx, start=off, nbytes=size)
        dma.sendchannel.wait()
        dma.recvchannel.wait()
    elapsed = time.perf_counter() - t0
    rx.invalidate()
    return elapsed, (n + chunk - 1) // chunk

## 1. Random-data BIST

In [ ]:
N = 640 * 480 * 3
tx = allocate(shape=(N,), dtype=np.uint8)
rx = allocate(shape=(N,), dtype=np.uint8)

fails = 0
for i in range(10):
    tx[:] = np.random.randint(0, 256, N, dtype=np.uint8)
    rx[:] = 0xEE
    t, nchunks = dma_loop(tx, rx)
    ok = np.array_equal(tx, rx)
    fails += not ok
    print(f'run {i}: {"ok  " if ok else "FAIL"} {N} B in {nchunks} transfers, {t*1e3:7.2f} ms, {N/t/1e6:6.1f} MB/s')
print('PASS' if fails == 0 else f'FAIL ({fails} runs)')

## 2. A webcam frame through the loopback

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
for _ in range(5):          # let auto-exposure settle
    ok, frame = cap.read()
cap.release()
assert ok, 'no frame from webcam'
print('frame', frame.shape, frame.dtype, frame.nbytes, 'bytes')

ftx = allocate(shape=frame.shape, dtype=np.uint8)
frx = allocate(shape=frame.shape, dtype=np.uint8)
ftx[:] = frame
frx[:] = 0
t, nchunks = dma_loop(ftx, frx)
print(f'{"identical" if np.array_equal(ftx, frx) else "MISMATCH"}: {nchunks} transfers, {t*1e3:.2f} ms')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].imshow(cv2.cvtColor(np.asarray(ftx), cv2.COLOR_BGR2RGB)); ax[0].set_title('sent (MM2S)')
ax[1].imshow(cv2.cvtColor(np.asarray(frx), cv2.COLOR_BGR2RGB)); ax[1].set_title('received (S2MM)')
for a in ax: a.axis('off')
plt.show()

In [ ]:
for b in (tx, rx, ftx, frx):
    b.freebuffer()